In [2]:
from pyspark.sql import SparkSession

# S3 버킷 경로
WAREHOUSE = "s3://iceberg-lab-smj/warehouse/"

# Iceberg + Glue 연동에 필요한 JAR 패키지
ICEBERG_VERSION = "1.5.0"
PACKAGES = ",".join([
    f"org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:{ICEBERG_VERSION}",
    "org.apache.iceberg:iceberg-aws-bundle:1.5.0",
])

spark = (
    SparkSession.builder
    .appName("IcebergGlueLab")
    .config("spark.jars.packages", PACKAGES)
    # Iceberg SQL 확장 (Time Travel 등)
    .config("spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    # Glue Catalog 연결
    .config("spark.sql.catalog.glue_catalog",
            "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.glue_catalog.catalog-impl",
            "org.apache.iceberg.aws.glue.GlueCatalog")
    # S3FileIO 설정
    .config("spark.sql.catalog.glue_catalog.io-impl",
            "org.apache.iceberg.aws.s3.S3FileIO")
    # 웨어하우스 경로
    .config("spark.sql.catalog.glue_catalog.warehouse", WAREHOUSE)
    # 리전 설정
    .config("spark.sql.catalog.glue_catalog.client.region",
            "ap-northeast-2")
    .getOrCreate()
)

print("Spark version:", spark.version)
print("Spark Session 생성 완료")

26/06/04 12:28:51 WARN Utils: Your hostname, sinminjaeui-MacBookPro.local resolves to a loopback address: 127.0.0.1; using 192.168.45.121 instead (on interface en0)
26/06/04 12:28:51 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/smj/.ivy2/cache
The jars for the packages stored in: /Users/smj/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
org.apache.iceberg#iceberg-aws-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-0e8ec12d-02e4-4b4d-a5d2-9e089e5acc19;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.5.0 in central
	found org.apache.iceberg#iceberg-aws-bundle;1.5.0 in central
:: resolution report :: resolve 70ms :: artifacts dl 2ms
	:: modules in use:
	org.apache.iceberg#iceberg-aws-bundle;1.5.0 from central in [default]
	org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.5.0 from central in [default]
	--------------

:: loading settings :: url = jar:file:/Users/smj/.pyenv/versions/3.11.9/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


26/06/04 12:28:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Spark version: 3.5.1
Spark Session 생성 완료


In [3]:
with open("./sql/ddl.sql", "r") as f:
    ddl = f.read()

spark.sql(ddl)
print("테이블 생성 완료")

spark.sql("DESCRIBE TABLE glue_catalog.ad_lakehouse.events").show(truncate=False)

테이블 생성 완료
+--------------+----------------+-------+
|col_name      |data_type       |comment|
+--------------+----------------+-------+
|event_id      |bigint          |NULL   |
|event_time    |timestamp       |NULL   |
|user_id       |string          |NULL   |
|amount        |double          |NULL   |
|              |                |       |
|# Partitioning|                |       |
|Part 0        |days(event_time)|       |
+--------------+----------------+-------+



In [4]:
spark.sql("""
    INSERT INTO glue_catalog.ad_lakehouse.events VALUES
        (1, TIMESTAMP '2026-04-10 09:00:00', 'u1', 100.0),
        (2, TIMESTAMP '2026-04-10 10:00:00', 'u2', 200.0)
""")

print("INSERT 1번 완료")

INSERT 1번 완료


In [5]:
spark.sql("""
    INSERT INTO glue_catalog.ad_lakehouse.events VALUES
        (3, TIMESTAMP '2026-04-11 11:00:00', 'u1', 150.0)
""")

print("INSERT 2번 완료")

INSERT 2번 완료


In [6]:
print("=== .snapshots ===")
spark.sql("""
    SELECT
        snapshot_id,
        parent_id,
        operation,
        committed_at
    FROM glue_catalog.ad_lakehouse.events.snapshots
""").show(truncate=False)

=== .snapshots ===
+-------------------+-------------------+---------+-----------------------+
|snapshot_id        |parent_id          |operation|committed_at           |
+-------------------+-------------------+---------+-----------------------+
|4564336367599048928|NULL               |append   |2026-06-04 12:29:11.076|
|615583982161795145 |4564336367599048928|append   |2026-06-04 12:29:46.168|
+-------------------+-------------------+---------+-----------------------+



In [7]:
print("=== .files (File Pruning 통계) ===")
spark.sql("""
    SELECT
        file_path,
        record_count,
        lower_bounds,
        upper_bounds
    FROM glue_catalog.ad_lakehouse.events.files
""").show(truncate=False)

=== .files (File Pruning 통계) ===
+---------------------------------------------------------------------------------------------------------------------------------+------------+--------------------------------------------------------------------------------------------------------------+--------------------------------------------------------------------------------------------------------------+
|file_path                                                                                                                        |record_count|lower_bounds                                                                                                  |upper_bounds                                                                                                  |
+---------------------------------------------------------------------------------------------------------------------------------+------------+-----------------------------------------------------------------------------------------

In [8]:
print("=== .history ===")
spark.sql("""
    SELECT *
    FROM glue_catalog.ad_lakehouse.events.history
""").show(truncate=False)

=== .history ===
+-----------------------+-------------------+-------------------+-------------------+
|made_current_at        |snapshot_id        |parent_id          |is_current_ancestor|
+-----------------------+-------------------+-------------------+-------------------+
|2026-06-04 12:29:11.076|4564336367599048928|NULL               |true               |
|2026-06-04 12:29:46.168|615583982161795145 |4564336367599048928|true               |
+-----------------------+-------------------+-------------------+-------------------+



In [10]:
print("=== 파티션별 통계 ===")
spark.sql("""
    SELECT
        partition,
        record_count,
        file_count
    FROM glue_catalog.ad_lakehouse.events.partitions
""").show(truncate=False)

=== 파티션별 통계 ===
+------------+------------+----------+
|partition   |record_count|file_count|
+------------+------------+----------+
|{2026-04-10}|2           |1         |
|{2026-04-11}|1           |1         |
+------------+------------+----------+

